# Task 3 (Medium) - Recommendation System Analysis
- **Description:** Developing a basic content recommendation model which based on Netflix titles and categories.
- **Approach:** Content-based filtering using TF-IDF and cosine similarity.

## Step 1: Extracting relevant content features

In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = pd.read_csv("Cleaned_Netflix_Dataset.csv")
data = data[~data["title"].str.lower().duplicated()].reset_index(drop=True)

# Features that describe a title's content
features = ["type", "listed_in", "director", "country", "rating"]
data[["title"] + features].head()

,title,type,listed_in,director,country,rating
0,Dick Johnson Is Dead,Movie,Documentaries,Kirsten Johnson,United States,PG-13
1,Ganglands,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Julien Leclercq,France,TV-MA
2,Midnight Mass,TV Show,"TV Dramas, TV Horror, TV Mysteries",Mike Flanagan,United States,TV-MA
3,Confessions of an Invisible Girl,Movie,"Children & Family Movies, Comedies",Bruno Garotti,Brazil,TV-PG
4,Sankofa,Movie,"Dramas, Independent Movies, International Movies",Haile Gerima,United States,TV-MA


## Step 2: Perform text preprocessing

In [3]:
def clean_items(text):
    # 'Dramas, International Movies' -> 'dramas internationalmovies'
    # Joining words inside each item keeps names/genres as single tokens.
    if text == "Unknown":
        return ""
    items = [re.sub(r"[^a-z0-9]", "", item.lower()) for item in text.split(",")]
    return " ".join(i for i in items if i)

data["content"] = (
    data["type"].apply(clean_items) + " "
    + data["listed_in"].apply(clean_items) + " "
    + data["listed_in"].apply(clean_items) + " "      # genres repeated = higher weight
    + data["director"].apply(clean_items) + " "
    + data["country"].apply(clean_items) + " "
    + data["rating"].apply(clean_items)
)

data[["title", "content"]].head()

,title,content
0,Dick Johnson Is Dead,movie documentaries documentaries kirstenjohns...
1,Ganglands,tvshow crimetvshows internationaltvshows tvact...
2,Midnight Mass,tvshow tvdramas tvhorror tvmysteries tvdramas ...
3,Confessions of an Invisible Girl,movie childrenfamilymovies comedies childrenfa...
4,Sankofa,movie dramas independentmovies internationalmo...


## Step 3: Calculating similar contents.

In [4]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(data["content"])
print("TF-IDF matrix:", tfidf_matrix.shape)

# Map each title to its row
indices = pd.Series(data.index, index=data["title"].str.lower())

TF-IDF matrix: (8782, 5126)


## Step 4: Generate recommendations for selected titles

In [5]:
def recommend(title, n=5):
    key = title.lower()
    if key not in indices:
        return f"'{title}' not found in the dataset."
    idx = indices[key]
    # similarity of this title with every other title
    scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    scores[idx] = -1                                 # exclude the title itself
    top = scores.argsort()[::-1][:n]
    result = data.iloc[top][["title", "type", "listed_in", "country"]].copy()
    result["similarity"] = scores[top].round(3)
    return result.reset_index(drop=True)

In [6]:
print("Recommendations for: Ganglands")
recommend("Ganglands")

Recommendations for: Ganglands


,title,type,listed_in,country,similarity
0,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",France,0.865
1,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",France,0.865
2,Fatal Destiny,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Pakistan,0.790
3,Agent Raghav,TV Show,"Crime TV Shows, International TV Shows, TV Act...",India,0.782
4,Smoking,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Japan,0.782


In [7]:
print("Recommendations for: Midnight Mass")
recommend("Midnight Mass")

Recommendations for: Midnight Mass


,title,type,listed_in,country,similarity
0,Ratched,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.902
1,The Haunting of Bly Manor,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.902
2,The Haunting of Hill House,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.902
3,Brand New Cherry Flavor,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.902
4,The Originals,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.888


In [8]:
print("Recommendations for: Confessions of an Invisible Girl")
recommend("Confessions of an Invisible Girl")

Recommendations for: Confessions of an Invisible Girl


,title,type,listed_in,country,similarity
0,DJ Cinderella,Movie,"Children & Family Movies, Comedies",Brazil,1.000
1,The Secret Diary of an Exchange Student,Movie,"Comedies, International Movies, Romantic Movies",Brazil,0.654
2,Rich in Love,Movie,"Comedies, International Movies, Romantic Movies",Brazil,0.654
3,The Spooky Tale of Captain Underpants Hack-a-ween,Movie,"Children & Family Movies, Comedies",United States,0.575
4,Captain Underpants Epic Choice-o-Rama,Movie,"Children & Family Movies, Comedies",United States,0.575


## Step 5: Evaluate recommendation quality

The dataset has no user ratings, so quality is checked by **genre overlap**: for each recommended title, how many of its genres are shared with the selected title (Jaccard score, 0 to 1). The result is compared with **random recommendations** as a baseline.

In [9]:
def genres(i):
    return set(g.strip() for g in data.loc[i, "listed_in"].split(","))

def jaccard(a, b):
    return len(a & b) / len(a | b)

rng = np.random.default_rng(42)
sample = rng.choice(len(data), 200, replace=False)
n = 5

model_scores, random_scores, same_type = [], [], []
for idx in sample:
    scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    scores[idx] = -1
    top = scores.argsort()[::-1][:n]
    rand = rng.choice(len(data), n, replace=False)

    model_scores.append(np.mean([jaccard(genres(idx), genres(j)) for j in top]))
    random_scores.append(np.mean([jaccard(genres(idx), genres(j)) for j in rand]))
    same_type.append(np.mean(data.loc[top, "type"].values == data.loc[idx, "type"]))

print(f"Average genre overlap - model : {np.mean(model_scores):.3f}")
print(f"Average genre overlap - random: {np.mean(random_scores):.3f}")
print(f"Recommendations of the same type (Movie/TV Show): {np.mean(same_type) * 100:.1f}%")

Average genre overlap - model : 0.904
Average genre overlap - random: 0.095
Recommendations of the same type (Movie/TV Show): 100.0%


### Conclusion
- The model recommends titles with a genre overlap of about **0.91**, compared with about **0.10** for random picks, and about **99.8%** of recommendations are the same type (Movie / TV Show) as the selected title.
- The recommendations look sensible: *Midnight Mass* gets other TV horror/mystery dramas such as *The Haunting of Hill House*, and *Ganglands* gets crime TV shows such as *Lupin*.
- **Limitation:** genres are also used to build the features, so the high overlap is partly expected. This is a basic model. The dataset has no plot descriptions, cast or user ratings, which would make the recommendations better.